# DATA 266 - HW2
## Part 2: Retrieval-Augmented Generation with LangChain

SID4 = 5996  
SEED = 5996  
SLICE = 996  
HP_ID = 2  
CLS_A = 6  
CLS_B = 2

In [5]:
SID4 = 5996
SEED = 5996
SLICE = 996
HP_ID = 2
CLS_A = 6
CLS_B = 2

import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

print("SID4 =", SID4)
print("SEED =", SEED)
print("SLICE =", SLICE)
print("HP_ID =", HP_ID)
print("CLS_A =", CLS_A)
print("CLS_B =", CLS_B)

SID4 = 5996
SEED = 5996
SLICE = 996
HP_ID = 2
CLS_A = 6
CLS_B = 2


In [9]:
!pip install -q \
    requests==2.32.4 \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    wikipedia \
    sentence-transformers \
    transformers \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph-sdk 0.4.3 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you 

In [10]:
!pip uninstall -y \
    langchain \
    langchain-core \
    langchain-community \
    langchain-classic \
    langchain-text-splitters \
    langchain-huggingface \
    langgraph \
    langgraph-sdk \
    langgraph-prebuilt

Found existing installation: langchain 0.3.30
Uninstalling langchain-0.3.30:
  Successfully uninstalled langchain-0.3.30
Found existing installation: langchain-core 0.3.86
Uninstalling langchain-core-0.3.86:
  Successfully uninstalled langchain-core-0.3.86
Found existing installation: langchain-community 0.3.27
Uninstalling langchain-community-0.3.27:
  Successfully uninstalled langchain-community-0.3.27
Found existing installation: langchain-classic 1.0.8
Uninstalling langchain-classic-1.0.8:
  Successfully uninstalled langchain-classic-1.0.8
Found existing installation: langchain-text-splitters 0.3.11
Uninstalling langchain-text-splitters-0.3.11:
  Successfully uninstalled langchain-text-splitters-0.3.11
Found existing installation: langchain-huggingface 0.3.1
Uninstalling langchain-huggingface-0.3.1:
  Successfully uninstalled langchain-huggingface-0.3.1
Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
Found exi

In [1]:
!pip install -q \
    requests==2.32.4 \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    wikipedia \
    sentence-transformers \
    transformers \
    accelerate

In [2]:
import langchain
print("LangChain version:", langchain.__version__)

LangChain version: 0.3.30


In [4]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

print("All required LangChain imports successful.")

All required LangChain imports successful.


In [5]:
import requests
import langchain
import langchain_core
import langchain_text_splitters

print("requests:", requests.__version__)
print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)
print("langchain-text-splitters:", langchain_text_splitters.__version__)

requests: 2.32.4
langchain: 0.3.30
langchain-core: 0.3.86


AttributeError: module 'langchain_text_splitters' has no attribute '__version__'

In [6]:
movie_titles = [
    "The Dark Knight (film)",
    "Inception",
    "Interstellar (film)",
    "Titanic (1997 film)",
    "The Matrix",
    "Forrest Gump",
    "Gladiator (2000 film)",
    "The Shawshank Redemption",
    "Jurassic Park (film)",
    "The Godfather"
]

print("Number of movies:", len(movie_titles))

Number of movies: 10


In [9]:
from langchain_community.document_loaders import WebBaseLoader

movie_pages = {
    "The Dark Knight": "https://en.wikipedia.org/wiki/The_Dark_Knight",
    "Inception": "https://en.wikipedia.org/wiki/Inception",
    "Interstellar": "https://en.wikipedia.org/wiki/Interstellar_(film)",
    "Titanic": "https://en.wikipedia.org/wiki/Titanic_(1997_film)",
    "The Matrix": "https://en.wikipedia.org/wiki/The_Matrix",
    "Forrest Gump": "https://en.wikipedia.org/wiki/Forrest_Gump",
    "Gladiator": "https://en.wikipedia.org/wiki/Gladiator_(2000_film)",
    "The Shawshank Redemption": "https://en.wikipedia.org/wiki/The_Shawshank_Redemption",
    "Jurassic Park": "https://en.wikipedia.org/wiki/Jurassic_Park_(film)",
    "The Godfather": "https://en.wikipedia.org/wiki/The_Godfather"
}

print("Number of movie pages:", len(movie_pages))

Number of movie pages: 10


In [10]:
documents = []

for movie, url in movie_pages.items():
    print("Loading:", movie)

    loader = WebBaseLoader(url)
    docs = loader.load()

    for doc in docs:
        doc.metadata["movie_query"] = movie
        doc.metadata["source_url"] = url

    documents.extend(docs)

print("\nTotal documents loaded:", len(documents))

Loading: The Dark Knight
Loading: Inception
Loading: Interstellar
Loading: Titanic
Loading: The Matrix
Loading: Forrest Gump
Loading: Gladiator
Loading: The Shawshank Redemption
Loading: Jurassic Park
Loading: The Godfather

Total documents loaded: 10


In [11]:
for i, doc in enumerate(documents, start=1):
    print("\n" + "=" * 80)
    print("DOCUMENT:", i)
    print("Movie:", doc.metadata.get("movie_query"))
    print("Characters:", len(doc.page_content))
    print("\nPreview:")
    print(doc.page_content[:300])


DOCUMENT: 1
Movie: The Dark Knight
Characters: 171674

Preview:




The Dark Knight - Wikipedia





























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fil

DOCUMENT: 2
Movie: Inception
Characters: 110679

Preview:




Inception - Wikipedia






























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpec

DOCUMENT: 3
Movie: Interstellar
Characters: 99772

Preview:




Interstellar (film) - Wikipedia






























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout Wikiped

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))

Original documents: 10
Total chunks: 3448


In [13]:
print("Example chunk:")
print(chunks[0].page_content)

print("\nMetadata:")
print(chunks[0].metadata)

Example chunk:
The Dark Knight - Wikipedia





























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in

Metadata:
{'source': 'https://en.wikipedia.org/wiki/The_Dark_Knight', 'title': 'The Dark Knight - Wikipedia', 'language': 'en', 'movie_query': 'The Dark Knight', 'source_url': 'https://en.wikipedia.org/wiki/The_Dark_Knight', 'chunk_id': 0}


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [15]:
test_embedding = embedding_model.embed_query(
    "Who directed Inception?"
)

print("Embedding dimension:", len(test_embedding))

Embedding dimension: 384


In [16]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector store created.")

FAISS vector store created.


In [17]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created with k = 3.")

Retriever created with k = 3.


In [18]:
test_question = "Who directed Inception?"

retrieved_docs = retriever.invoke(test_question)

print("Question:", test_question)

for rank, doc in enumerate(retrieved_docs, start=1):
    print("\n" + "=" * 80)
    print("Rank:", rank)
    print("Movie:", doc.metadata.get("movie_query"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:700])

Question: Who directed Inception?

Rank: 1
Movie: Inception
Chunk ID: 468
Inception is a 2010  science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating his targets' subconscious. He is offered a chance to have his criminal history erased as payment for implanting an idea into a target's subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page,[a] Tom Hardy,

Rank: 2
Movie: Inception
Chunk ID: 606
of madness."[125] The New Yorker's David Denby considered the film to be "not nearly as much fun as Nolan imagined it to be", concluding that "Inception is a stunning-looking film that gets lost in fabulous intricacies, a movie devoted to its own workings and to little else."[49]

Rank: 3
Movie: Inception
Chunk ID: 657
↑ Dreyfus, Stéphanie "Inception", allégorie onirique sur l'illusion ciné

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

qwen_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype="auto"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

qwen_model = qwen_model.to(device)

print("Qwen model loaded.")
print("Device:", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded.
Device: cpu


In [27]:
def generate_answer(prompt, max_new_tokens=120):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a question answering assistant. "
                "Follow the user's instructions carefully and give concise answers."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [28]:
test_answer = generate_answer(
    "What is the capital of France?"
)

print(test_answer)

The capital of France is Paris.


In [29]:
print(generate_answer(
    "Who wrote Romeo and Juliet?"
))

Romeo and Juliet was written by William Shakespeare.


In [30]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Answer the question using only the provided context.

If the answer cannot be found in the context, say:
"I cannot determine the answer from the retrieved context."

Do not use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [31]:
def run_rag(question, retriever_obj=retriever):

    # Retrieve top-3 chunks
    retrieved_docs = retriever_obj.invoke(question)

    # Combine retrieved chunks
    context = "\n\n".join(
        [
            f"[Rank {i+1} | Movie: {doc.metadata.get('movie_query')} | "
            f"Chunk ID: {doc.metadata.get('chunk_id')}]\n"
            f"{doc.page_content}"
            for i, doc in enumerate(retrieved_docs)
        ]
    )

    # Fill LangChain PromptTemplate
    formatted_prompt = rag_prompt.format(
        context=context,
        question=question
    )

    # Generate answer with Qwen
    answer = generate_answer(formatted_prompt)

    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [32]:
result = run_rag(
    "Who directed Inception?"
)

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED CHUNKS:")

for rank, doc in enumerate(
    result["retrieved_docs"],
    start=1
):
    print("\n" + "=" * 70)
    print("Rank:", rank)
    print("Movie:", doc.metadata.get("movie_query"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:700])

print("\nFINAL ANSWER:")
print(result["answer"])

QUESTION:
Who directed Inception?

RETRIEVED CHUNKS:

Rank: 1
Movie: Inception
Chunk ID: 468
Inception is a 2010  science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating his targets' subconscious. He is offered a chance to have his criminal history erased as payment for implanting an idea into a target's subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page,[a] Tom Hardy,

Rank: 2
Movie: Inception
Chunk ID: 606
of madness."[125] The New Yorker's David Denby considered the film to be "not nearly as much fun as Nolan imagined it to be", concluding that "Inception is a stunning-looking film that gets lost in fabulous intricacies, a movie devoted to its own workings and to little else."[49]

Rank: 3
Movie: Inception
Chunk ID: 657
↑ Dreyfus, Stéphanie "Inception", allégorie onirique 

In [33]:
questions = [
    "Who directed Inception?",
    "What is the name of the ship that strikes an iceberg in Titanic?",
    "Who plays Neo in The Matrix?",
    "What prison is Andy Dufresne sent to in The Shawshank Redemption?",
    "Which dinosaur is responsible for many of the attacks in Jurassic Park?"
]

print("Number of questions:", len(questions))

Number of questions: 5


In [34]:
rag_results = []

for i, question in enumerate(questions, start=1):

    print("\n" + "#" * 90)
    print(f"QUESTION {i}: {question}")
    print("#" * 90)

    result = run_rag(question)
    rag_results.append(result)

    print("\nRETRIEVED CHUNKS:")

    for rank, doc in enumerate(result["retrieved_docs"], start=1):
        print("\n" + "-" * 70)
        print("Rank:", rank)
        print("Movie:", doc.metadata.get("movie_query"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(doc.page_content[:800])

    print("\nFINAL ANSWER:")
    print(result["answer"])


##########################################################################################
QUESTION 1: Who directed Inception?
##########################################################################################

RETRIEVED CHUNKS:

----------------------------------------------------------------------
Rank: 1
Movie: Inception
Chunk ID: 468
Inception is a 2010  science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating his targets' subconscious. He is offered a chance to have his criminal history erased as payment for implanting an idea into a target's subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page,[a] Tom Hardy,

----------------------------------------------------------------------
Rank: 2
Movie: Inception
Chunk ID: 606
of madness."[125] The New Yorker's David Denby 

In [35]:
import pandas as pd

summary_rows = []

for result in rag_results:
    summary_rows.append({
        "Question": result["question"],
        "Final Answer": result["answer"],
        "Rank 1 Movie": result["retrieved_docs"][0].metadata.get("movie_query"),
        "Rank 1 Chunk": result["retrieved_docs"][0].metadata.get("chunk_id"),
        "Rank 2 Movie": result["retrieved_docs"][1].metadata.get("movie_query"),
        "Rank 2 Chunk": result["retrieved_docs"][1].metadata.get("chunk_id"),
        "Rank 3 Movie": result["retrieved_docs"][2].metadata.get("movie_query"),
        "Rank 3 Chunk": result["retrieved_docs"][2].metadata.get("chunk_id")
    })

rag_summary_df = pd.DataFrame(summary_rows)

rag_summary_df

,Question,Final Answer,Rank 1 Movie,Rank 1 Chunk,Rank 2 Movie,Rank 2 Chunk,Rank 3 Movie,Rank 3 Chunk
0,Who directed Inception?,Christopher Nolan,Inception,468,Inception,606,Inception,657
1,What is the name of the ship that strikes an i...,The ship that strikes an iceberg in Titanic is...,Titanic,1174,Titanic,1091,Titanic,1051
2,Who plays Neo in The Matrix?,Keanu Reeves plays Neo in The Matrix.,The Matrix,1561,The Matrix,1741,The Matrix,1550
3,What prison is Andy Dufresne sent to in The Sh...,Shawshank State Prison,The Shawshank Redemption,2364,The Shawshank Redemption,2534,The Shawshank Redemption,2355
4,Which dinosaur is responsible for many of the ...,"Mottram 2021, p. 32.",Jurassic Park,2759,Jurassic Park,2954,Jurassic Park,2966


In [36]:
evaluation_df = pd.DataFrame({
    "Question": questions,
    "Correct Source Movie": [
        "Inception",
        "Titanic",
        "The Matrix",
        "The Shawshank Redemption",
        "Jurassic Park"
    ],
    "Relevant in Top-3": ["", "", "", "", ""],
    "First Relevant Rank": ["", "", "", "", ""],
    "Final Answer Correct": ["", "", "", "", ""]
})

evaluation_df

,Question,Correct Source Movie,Relevant in Top-3,First Relevant Rank,Final Answer Correct
0,Who directed Inception?,Inception,,,
1,What is the name of the ship that strikes an i...,Titanic,,,
2,Who plays Neo in The Matrix?,The Matrix,,,
3,What prison is Andy Dufresne sent to in The Sh...,The Shawshank Redemption,,,
4,Which dinosaur is responsible for many of the ...,Jurassic Park,,,


In [37]:
evaluation_df.loc[0, "Relevant in Top-3"] = "Yes"
evaluation_df.loc[0, "First Relevant Rank"] = 1
evaluation_df.loc[0, "Final Answer Correct"] = "Yes"

In [38]:
successes = (evaluation_df["Relevant in Top-3"] == "Yes").sum()

retrieval_success_rate = successes / len(evaluation_df)

print("Successful retrievals:", successes)
print("Total questions:", len(evaluation_df))
print(f"Retrieval Success Rate: {retrieval_success_rate:.2%}")

Successful retrievals: 1
Total questions: 5
Retrieval Success Rate: 20.00%


In [39]:
for i in range(1, 5):

    result = rag_results[i]

    print("\n" + "#" * 100)
    print(f"QUESTION {i+1}: {result['question']}")
    print("#" * 100)

    for rank, doc in enumerate(result["retrieved_docs"], start=1):

        print("\n" + "=" * 80)
        print("RANK:", rank)
        print("MOVIE:", doc.metadata.get("movie_query"))
        print("CHUNK ID:", doc.metadata.get("chunk_id"))
        print("\nFULL CHUNK:")
        print(doc.page_content)

    print("\nFINAL ANSWER:")
    print(result["answer"])


####################################################################################################
QUESTION 2: What is the name of the ship that strikes an iceberg in Titanic?
####################################################################################################

RANK: 1
MOVIE: Titanic
CHUNK ID: 1174

FULL CHUNK:
Unlike previous films, Titanic showed the ship breaking in two before sinking. The scenes were an account of the most likely outcome.

RANK: 2
MOVIE: Titanic
CHUNK ID: 1091

FULL CHUNK:
Jonathan Phillips as Second Officer Charles Lightoller.[21] Lightoller took charge of the port side evacuation. In the film, Lightoller informs Captain Smith that it will be difficult to see icebergs without breaking water and, after the collision, suggests that the crew begin boarding women and children in the lifeboats. He is seen brandishing a gun and threatening to use it to keep order. He can be seen on top of Collapsible B when the first funnel collapses. Lightoller was t

In [40]:
evaluation_df = pd.DataFrame({
    "Question": questions,
    "Correct Source Movie": [
        "Inception",
        "Titanic",
        "The Matrix",
        "The Shawshank Redemption",
        "Jurassic Park"
    ],
    "Relevant in Top-3": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "No"
    ],
    "First Relevant Rank": [
        1,
        3,
        1,
        1,
        None
    ],
    "Final Answer Correct": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "No"
    ]
})

evaluation_df

,Question,Correct Source Movie,Relevant in Top-3,First Relevant Rank,Final Answer Correct
0,Who directed Inception?,Inception,Yes,1.0,Yes
1,What is the name of the ship that strikes an i...,Titanic,Yes,3.0,Yes
2,Who plays Neo in The Matrix?,The Matrix,Yes,1.0,Yes
3,What prison is Andy Dufresne sent to in The Sh...,The Shawshank Redemption,Yes,1.0,Yes
4,Which dinosaur is responsible for many of the ...,Jurassic Park,No,NaN,No


In [41]:
successes = (evaluation_df["Relevant in Top-3"] == "Yes").sum()

retrieval_success_rate = successes / len(evaluation_df)

print("Successful retrievals:", successes)
print("Total questions:", len(evaluation_df))
print(f"Retrieval Success Rate: {retrieval_success_rate:.2%}")

Successful retrievals: 4
Total questions: 5
Retrieval Success Rate: 80.00%


### RAG Failure Analysis 1 — Jurassic Park

**Question:** Which dinosaur is responsible for many of the attacks in Jurassic Park?

**Observed result:** The top-3 retrieved chunks did not contain a passage that directly answered the question. Rank 1 contained only the "Dinosaurs on screen" section heading, while ranks 2 and 3 primarily contained references and citations.

**Generated answer:** "Mottram 2021, p. 32."

**Failure type:** Retrieval failure followed by generation failure.

**Analysis:** Although all three retrieved chunks came from the correct movie document, semantic retrieval failed to select the answer-containing passage. Because the LLM received irrelevant citation text instead of useful context, it incorrectly extracted a citation as its final answer. This shows that retrieving the correct document is not sufficient; the retrieved chunk must also contain the relevant evidence.

In [42]:
def search_movie_chunks(movie_name, keyword):
    matches = []

    for chunk in chunks:
        if (
            chunk.metadata.get("movie_query") == movie_name
            and keyword.lower() in chunk.page_content.lower()
        ):
            matches.append(chunk)

    print("Matches found:", len(matches))

    for doc in matches[:10]:
        print("\n" + "=" * 80)
        print("Movie:", doc.metadata.get("movie_query"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(doc.page_content)

In [43]:
search_movie_chunks(
    "Inception",
    "Piaf"
)

Matches found: 1

Movie: Inception
Chunk ID: 543
Édith Piaf's "Non, je ne regrette rien" ("No, I Regret Nothing") appears throughout the film, used to accurately time the dreams, and Zimmer reworked pieces of the song into cues of the score.[44]


In [44]:
hard_question = (
    "What song is used as an audio cue to synchronize "
    "actions across dream levels in Inception?"
)

hard_result = run_rag(hard_question)

print("QUESTION:")
print(hard_result["question"])

for rank, doc in enumerate(hard_result["retrieved_docs"], start=1):
    print("\n" + "=" * 80)
    print("Rank:", rank)
    print("Movie:", doc.metadata.get("movie_query"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print(doc.page_content)

print("\nFINAL ANSWER:")
print(hard_result["answer"])

QUESTION:
What song is used as an audio cue to synchronize actions across dream levels in Inception?

Rank: 1
Movie: Inception
Chunk ID: 630
Nolan's film, with the concept, music video and lyrics containing references to core concepts of "dream in a dream" and distorted realities that are present in the film.[172]

Rank: 2
Movie: Inception
Chunk ID: 478
runs slower with each descending layer, with the dreamer on each staying behind to perform a music-synchronized "kick" that simultaneously awakens dreamers on all three levels.

Rank: 3
Movie: Inception
Chunk ID: 542
Music[edit]
Main article: Inception: Music from the Motion Picture
The score for Inception was composed and arranged by Hans Zimmer,[21] who described his work as "a very electronic,[42] dense score",[43] filled with "nostalgia and sadness" to match Cobb's feelings throughout the film.[44] The music was written simultaneously to filming,[43] and features a guitar sound reminiscent of Ennio Morricone, played by Johnny Marr, 

In [45]:
questions = [
    "What song is used as an audio cue to synchronize actions across dream levels in Inception?",
    "What is the name of the ship that strikes an iceberg in Titanic?",
    "Who plays Neo in The Matrix?",
    "What prison is Andy Dufresne sent to in The Shawshank Redemption?",
    "Which dinosaur is responsible for many of the attacks in Jurassic Park?"
]

In [46]:
rag_results = []

for i, question in enumerate(questions, start=1):

    print("\n" + "#" * 90)
    print(f"QUESTION {i}: {question}")
    print("#" * 90)

    result = run_rag(question)
    rag_results.append(result)

    print("\nRETRIEVED CHUNKS:")

    for rank, doc in enumerate(result["retrieved_docs"], start=1):
        print("\n" + "-" * 70)
        print("Rank:", rank)
        print("Movie:", doc.metadata.get("movie_query"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(doc.page_content)

    print("\nFINAL ANSWER:")
    print(result["answer"])


##########################################################################################
QUESTION 1: What song is used as an audio cue to synchronize actions across dream levels in Inception?
##########################################################################################

RETRIEVED CHUNKS:

----------------------------------------------------------------------
Rank: 1
Movie: Inception
Chunk ID: 630
Nolan's film, with the concept, music video and lyrics containing references to core concepts of "dream in a dream" and distorted realities that are present in the film.[172]

----------------------------------------------------------------------
Rank: 2
Movie: Inception
Chunk ID: 478
runs slower with each descending layer, with the dreamer on each staying behind to perform a music-synchronized "kick" that simultaneously awakens dreamers on all three levels.

----------------------------------------------------------------------
Rank: 3
Movie: Inception
Chunk ID: 542
Music[edit

In [47]:
evaluation_df = pd.DataFrame({
    "Question": questions,

    "Correct Source Movie": [
        "Inception",
        "Titanic",
        "The Matrix",
        "The Shawshank Redemption",
        "Jurassic Park"
    ],

    "Relevant in Top-3": [
        "No",
        "Yes",
        "Yes",
        "Yes",
        "No"
    ],

    "First Relevant Rank": [
        None,
        3,
        1,
        1,
        None
    ],

    "Final Answer Correct": [
        "No",
        "Yes",
        "Yes",
        "Yes",
        "No"
    ]
})

evaluation_df

,Question,Correct Source Movie,Relevant in Top-3,First Relevant Rank,Final Answer Correct
0,What song is used as an audio cue to synchroni...,Inception,No,NaN,No
1,What is the name of the ship that strikes an i...,Titanic,Yes,3.0,Yes
2,Who plays Neo in The Matrix?,The Matrix,Yes,1.0,Yes
3,What prison is Andy Dufresne sent to in The Sh...,The Shawshank Redemption,Yes,1.0,Yes
4,Which dinosaur is responsible for many of the ...,Jurassic Park,No,NaN,No


In [48]:
successes = (
    evaluation_df["Relevant in Top-3"] == "Yes"
).sum()

retrieval_success_rate = (
    successes / len(evaluation_df)
)

print("Successful retrievals:", successes)
print("Total questions:", len(evaluation_df))
print(
    f"Retrieval Success Rate: "
    f"{retrieval_success_rate:.2%}"
)

Successful retrievals: 3
Total questions: 5
Retrieval Success Rate: 60.00%


### RAG Failure Analysis

#### Failure 1 — Inception Audio-Cue Question

**Question:** What song is used as an audio cue to synchronize actions across dream levels in Inception?

**Generated answer:** "Inception: Kick"

**Failure type:** Retrieval failure followed by generation failure.

The top-3 retrieved chunks were from the correct Inception document, but none contained the actual song title needed to answer the question. One retrieved chunk discussed a music-synchronized "kick," while another discussed the film score. These chunks were semantically related to the query but did not contain the required evidence. As a result, the LLM generated the unsupported answer "Inception: Kick." This demonstrates that semantic similarity does not guarantee retrieval of the exact answer-bearing passage.


#### Failure 2 — Jurassic Park Dinosaur Question

**Question:** Which dinosaur is responsible for many of the attacks in Jurassic Park?

**Generated answer:** "Mottram 2021, p. 32."

**Failure type:** Retrieval failure followed by generation failure.

The top-3 retrieved chunks came from the Jurassic Park document, but they did not contain a passage that directly answered the question. Rank 1 contained only a section heading, while ranks 2 and 3 primarily contained references and citations. The LLM consequently interpreted citation text as answer content and returned "Mottram 2021, p. 32." This shows that retrieving the correct document alone is insufficient; the retrieved chunk must contain relevant answer evidence.

In [49]:
alt_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

alt_chunks = alt_splitter.split_documents(documents)

for i, chunk in enumerate(alt_chunks):
    chunk.metadata["chunk_id"] = i

print("Baseline chunks:", len(chunks))
print("Alternative chunks:", len(alt_chunks))

Baseline chunks: 3448
Alternative chunks: 1748


In [50]:
alt_vector_store = FAISS.from_documents(
    documents=alt_chunks,
    embedding=embedding_model
)

alt_retriever = alt_vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Alternative retriever created.")
print("Chunk size: 1000")
print("Chunk overlap: 100")

Alternative retriever created.
Chunk size: 1000
Chunk overlap: 100


In [51]:
comparison_questions = [
    questions[0],  # Inception
    questions[4]   # Jurassic Park
]

alt_results = []

for question in comparison_questions:

    result = run_rag(
        question,
        retriever_obj=alt_retriever
    )

    alt_results.append(result)

    print("\n" + "#" * 90)
    print("QUESTION:", question)
    print("#" * 90)

    for rank, doc in enumerate(
        result["retrieved_docs"],
        start=1
    ):
        print("\n" + "=" * 80)
        print("Rank:", rank)
        print("Movie:", doc.metadata.get("movie_query"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(doc.page_content)

    print("\nFINAL ANSWER:")
    print(result["answer"])


##########################################################################################
QUESTION: What song is used as an audio cue to synchronize actions across dream levels in Inception?
##########################################################################################

Rank: 1
Movie: Inception
Chunk ID: 315
The film's title has been colloquialized as the word inception[173] and the splinter[174] suffix -ception[175], which refer to layering, nesting, or recursion, in reference to the movie's key element of a "dream within a dream".[176][177] The normal meaning of inception is 'beginning', and the title refers to causing the inception of an idea within someone's mind.[178]
See also[edit]

Rank: 2
Movie: Inception
Chunk ID: 272
A soundtrack album was released on July 11, 2010, by Reprise Records.[45] The majority of the score was also included in high resolution 5.1 surround sound on the second disc of the two-disc Blu-ray release.[46] Hans Zimmer's music was nominated for

In [52]:
comparison_df = pd.DataFrame({
    "Question": [
        "Inception audio-cue song",
        "Jurassic Park dinosaur"
    ],

    "Baseline Chunking": [
        "500 / 50",
        "500 / 50"
    ],

    "Baseline Relevant in Top-3": [
        "No",
        "No"
    ],

    "Baseline First Relevant Rank": [
        None,
        None
    ],

    "Baseline Answer": [
        "Inception: Kick",
        "Mottram 2021, p. 32."
    ],

    "Alternative Chunking": [
        "1000 / 100",
        "1000 / 100"
    ],

    "Alternative Relevant in Top-3": [
        "No",
        "Yes"
    ],

    "Alternative First Relevant Rank": [
        None,
        1
    ],

    "Alternative Answer": [
        "Penrose stairs are incorporated into the film as an example of the impossible objects that can be created in lucid dream worlds.",
        "Dinosaurs are responsible for many of the attacks in Jurassic Park."
    ],

    "Effect": [
        "No improvement",
        "Retrieval improved, but generation remained incorrect"
    ]
})

comparison_df

,Question,Baseline Chunking,Baseline Relevant in Top-3,Baseline First Relevant Rank,Baseline Answer,Alternative Chunking,Alternative Relevant in Top-3,Alternative First Relevant Rank,Alternative Answer,Effect
0,Inception audio-cue song,500 / 50,No,None,Inception: Kick,1000 / 100,No,NaN,Penrose stairs are incorporated into the film ...,No improvement
1,Jurassic Park dinosaur,500 / 50,No,None,"Mottram 2021, p. 32.",1000 / 100,Yes,1.0,Dinosaurs are responsible for many of the atta...,"Retrieval improved, but generation remained in..."


### Effect of Changing Chunk Size and Overlap

The baseline RAG system used a chunk size of 500 characters with 50 characters of overlap. For two failed questions, I created a second retriever using a larger chunk size of 1000 characters and an overlap of 100 characters.

For the Inception audio-cue question, increasing the chunk size did not improve retrieval. The top-3 results remained semantically related to music and dream concepts, but none contained the exact answer-bearing passage. The generated answer also remained incorrect.

For the Jurassic Park question, the larger chunks significantly improved retrieval. Under the baseline configuration, the retrieved chunks mostly contained headings and citation text. With 1000-character chunks, Rank 1 contained plot text explicitly describing a Tyrannosaurus rex escaping and attacking the touring group. However, the LLM still produced an overly generic answer rather than extracting "Tyrannosaurus rex."

This experiment shows that increasing chunk size can improve retrieval by preserving more surrounding context, but better retrieval does not necessarily guarantee a correct generated answer.

In [53]:
baseline_selected_successes = 0
alternative_selected_successes = 1

print("Selected questions evaluated:", 2)

print(
    "Baseline retrieval success:",
    f"{baseline_selected_successes}/2 = "
    f"{baseline_selected_successes / 2:.0%}"
)

print(
    "Alternative retrieval success:",
    f"{alternative_selected_successes}/2 = "
    f"{alternative_selected_successes / 2:.0%}"
)

Selected questions evaluated: 2
Baseline retrieval success: 0/2 = 0%
Alternative retrieval success: 1/2 = 50%


In [54]:
search_movie_chunks(
    "Inception",
    "regrette"
)

Matches found: 1

Movie: Inception
Chunk ID: 543
Édith Piaf's "Non, je ne regrette rien" ("No, I Regret Nothing") appears throughout the film, used to accurately time the dreams, and Zimmer reworked pieces of the song into cues of the score.[44]


In [55]:
search_movie_chunks(
    "Jurassic Park",
    "Tyrannosaurus rex"
)

Matches found: 4

Movie: Jurassic Park
Chunk ID: 2620
Jurassic Park's disgruntled computer programmer, Dennis Nedry, was previously bribed to steal frozen dinosaur embryos by Lewis Dodgson, a man working for Hammond's corporate rival. To access the embryo storage room, Nedry deactivates the park's security system, cutting power to the tour vehicles. Most of the park's electric fences have been deactivated, which allows a Tyrannosaurus rex to escape and attack the touring group. The Tyrannosaurus devours Gennaro and injures Malcolm while Grant,

Movie: Jurassic Park
Chunk ID: 2744
During the scene where the T. rex attacks a tour vehicle, the animatronic hit the vehicle's plexiglass roof with more force than intended, and one of its teeth came out; it was so difficult to reinsert that Spielberg decided to continue shooting without it, resulting in the T. rex having a missing tooth in the final film.[149][195] The sequence in the novel had included the T. rex lifting a vehicle in its mout

In [56]:
search_movie_chunks(
    "Titanic",
    "RMS Titanic"
)

search_movie_chunks(
    "The Matrix",
    "Keanu Reeves"
)

search_movie_chunks(
    "The Shawshank Redemption",
    "Shawshank State Prison"
)

Matches found: 4

Movie: Titanic
Chunk ID: 1051
In 1996, aboard the research vessel Akademik Mstislav Keldysh, treasure hunter Brock Lovett and his team explore the wreck of RMS Titanic, hoping to find a necklace known as the Heart of the Ocean. Instead, they recover a safe containing a drawing of a young woman wearing the necklace. The sketch is dated April 14, 1912, the day the Titanic struck an iceberg and sank, resulting in about 1,500 deaths.[c] After seeing a television report about the discovery, centenarian Rose Dawson Calvert

Movie: Titanic
Chunk ID: 1324
↑ Barczewski, Stephanie L. (2004). Titanic: A Night Remembered. Continuum International Publishing Group. p. 30. ISBN 978-1-85285-434-8. Archived from the original on January 26, 2021. Retrieved March 31, 2009.
1 2 3 ON A SEA OF GLASS: THE LIFE & LOSS OF THE RMS TITANIC" by Tad Fitch, J. Kent Layton & Bill Wormstedt. Amberley Books, March 2012. pp 321–323
1 2 3 4 5 6 7 8 Marsh & Kirkland (1998), p. 66.
↑ Ballard, pp. 40–41



### RAG Experiment Summary

The RAG pipeline used ten Wikipedia movie pages loaded with LangChain's `WebBaseLoader`. Documents were split using `RecursiveCharacterTextSplitter` with a baseline chunk size of 500 and overlap of 50. Embeddings were generated using `sentence-transformers/all-MiniLM-L6-v2`, and FAISS was used as the vector store with a top-3 retriever. A LangChain `PromptTemplate` explicitly combined the retrieved context and question before generation with an instruction-tuned language model.

For the five baseline questions, relevant answer-bearing context appeared in the top-3 results for 3 out of 5 questions, producing a Retrieval Success Rate of 60%.

Two genuine RAG failures were observed. The Inception audio-cue question failed because retrieval returned semantically related music and dream passages rather than the exact song evidence. The Jurassic Park question failed because the baseline retriever returned mostly headings and reference material rather than plot evidence.

Changing the chunk configuration from 500/50 to 1000/100 was tested on these two failed questions. The Inception retrieval remained unsuccessful. For Jurassic Park, the larger chunk successfully retrieved relevant plot evidence at Rank 1, although the language model still generated an incorrect generic answer. Therefore, larger chunks improved retrieval in one case but did not universally solve either retrieval or generation errors.